<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/26_tool_augmented_rag/tool_augmented_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool-Augmented RAG

This notebook builds a RAG system that can use external tools like a calculator along with document retrieval.

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
documents = [
    "Elon Musk founded SpaceX.",
    "Tesla is an electric vehicle company.",
    "SpaceX works in space exploration.",
    "Elon Musk is the CEO of Tesla."
]

df = pd.DataFrame({"text": documents})

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(df['text'].tolist())

In [ ]:
def retrieve(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return df.iloc[top_indices]

In [ ]:
def calculator(query):
    try:
        return str(eval(query))
    except:
        return None

In [ ]:
def needs_calculation(query):
    return any(op in query for op in ["+", "-", "*", "/"])

In [ ]:
def extract_answer(context, query, calc_result=None):
    context_lower = context.lower()

    answer_parts = []

    if calc_result:
        answer_parts.append(calc_result)

    if "elon musk" in query.lower():
        if "spacex" in context_lower:
            answer_parts.append("SpaceX")

    return " and ".join(answer_parts)

In [ ]:
def tool_augmented_rag(query):
    print("🔹 Query:", query)

    # Step 1: Tool usage
    calc_result = None
    if needs_calculation(query):
        expression = "".join([c for c in query if c in "0123456789+-*/"])
        calc_result = calculator(expression)
        print("\n🧮 Calculator Result:", calc_result)

    # Step 2: Retrieval
    docs = retrieve(query)
    context = " ".join(docs['text'].tolist())
    print("\n📄 Retrieved Context:\n", context)

    # Step 3: Final Answer
    answer = extract_answer(context, query, calc_result)
    print("\n✅ Final Answer:", answer)

    return answer

In [11]:
tool_augmented_rag("What is 25*4 and which company did Elon Musk found?")

🔹 Query: What is 25*4 and which company did Elon Musk found?

🧮 Calculator Result: 100

📄 Retrieved Context:
 Elon Musk founded SpaceX. Elon Musk is the CEO of Tesla.

✅ Final Answer: 100 and SpaceX


'100 and SpaceX'